In [1]:
import csv, time, json, ast
from seleniumbase import Driver
from pprint import pprint
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import os
import shutil
import csv

In [2]:
brand = 'TMD'
browser = Driver(uc=True, incognito=True)
browser_wait = WebDriverWait(browser, 60)
# browser.maximize_window()

In [3]:
browser.get(f'https://www.arnoldclarkautoparts.com')

In [20]:
header = ['SKU', 'Link']
with open(fr"comp_scrape.csv", 'a', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow(header)
    df1 = pd.read_csv(fr"C:\Users\admin\Documents\Scrap draft\Mintex\Mintex-scrape.csv", dtype = str)
    display(df1)
    for index, row in df1.iterrows():
        # try:
        sku = row['PN']
        browser.get(f'https://www.arnoldclarkautoparts.com/search?type=product&options%5Bprefix%5D=last&options%5Bunavailable_products%5D=&q={sku}')
        time.sleep(1)
    
        product = browser.find_element(by=By.CSS_SELECTOR, value='.product-item__title').get_attribute('href')
        writer.writerow([sku, product])
        print([sku, product])
            
        # except Exception as e:
            # print(f'{str(e).splitlines()[0]}')

,PN,title,sku,clean_desc,clean_spec,comp_list,image_list,Barcode,Weight (kgs)
0,MDB1934,"Brake Pad Set, disc brake",MDB1934,<span>Brake System</span>:<span>Teves</span>; ...,NaN,[],['https://www.brakebook.com/pim/upload/TMDImag...,5.03E+12,1.8
1,MDB2068,"Brake Pad Set, disc brake",MDB2068,<span>Brake System</span>:<span>Teves</span>; ...,NaN,[],['https://www.brakebook.com/pim/upload/TMDImag...,5.03E+12,1.98
2,MDB2089,"Brake Pad Set, disc brake",MDB2089,<span>Brake System</span>:<span>Brembo</span>;...,NaN,[],['https://www.brakebook.com/pim/upload/TMDImag...,5.03E+12,1.32
3,MDB2122,"Brake Pad Set, disc brake",MDB2122,<span>Brake System</span>:<span>Sumitomo</span...,NaN,[],['https://www.brakebook.com/pim/upload/TMDImag...,5.03E+12,1.72
4,MDB2123,"Brake Pad Set, disc brake",MDB2123,<span>Brake System</span>:<span>Teves</span>; ...,NaN,[],['https://www.brakebook.com/pim/upload/TMDImag...,5.03E+12,1.44
...,...,...,...,...,...,...,...,...,...
551,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
552,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
553,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
554,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


['MDB1934', 'https://www.arnoldclarkautoparts.com/products/mintex-car-brake-pads-mdb1936?_pos=1&_sid=c04347c6a&_ss=r']
['MDB2068', 'https://www.arnoldclarkautoparts.com/products/mintex-car-brake-pads-mdb2068?_pos=1&_sid=ff00508ac&_ss=r']
['MDB2089', 'https://www.arnoldclarkautoparts.com/products/mintex-car-brake-pads-mdb2089?_pos=1&_sid=a7ccb90ed&_ss=r']
['MDB2122', 'https://www.arnoldclarkautoparts.com/products/mintex-car-brake-pads-mdb2122?_pos=1&_sid=0027a5220&_ss=r']
['MDB2123', 'https://www.arnoldclarkautoparts.com/products/mintex-car-brake-pads-mdb2123?_pos=1&_sid=6a915a732&_ss=r']
['MDB2167', 'https://www.arnoldclarkautoparts.com/products/mintex-car-brake-pads-mdb2168?_pos=1&_sid=6f2268d72&_ss=r']
['MDB2168', 'https://www.arnoldclarkautoparts.com/products/mintex-car-brake-pads-mdb2168?_pos=1&_sid=115073457&_ss=r']
['MDB2188', 'https://www.arnoldclarkautoparts.com/products/mintex-car-brake-pads-mdb2188?_pos=1&_sid=c7c20ede2&_ss=r']
['MDB2189', 'https://www.arnoldclarkautoparts.co

In [21]:
header = ['link', 'sku', 'sku2', 'title', 'vendor', 'clean_desc', 'clean_features', 'clean_com', 'prices', 'image_list']
with open(f'mintex_comp_scrape.csv', 'a', newline='', encoding='utf-8-sig') as f:
    writer = csv.writer(f)
    writer.writerow(header)

    df1 = pd.read_csv(r'comp_scrape.csv', dtype=str)
    for index, row in df1.iterrows():
        link = row['Link']
        sku = row['SKU']

        browser.get(link)
        time.sleep(1)
        try:
            title = browser.find_element(by=By.CSS_SELECTOR, value='h1.product-meta__title').get_attribute('innerText').strip()
            sku2 = browser.find_element(by=By.CSS_SELECTOR, value='.product-meta__sku-number').get_attribute('innerText').strip()
            vendor = browser.find_element(by=By.CSS_SELECTOR, value='.product-meta__vendor').get_attribute('innerText').strip()

            try:
                desc = browser.find_element(by=By.CSS_SELECTOR, value='.product-block-list__item--description p').get_attribute('innerHTML').split('<strong>OEM References')[0].split('Paste:')[0]
                clean_desc = re.sub(r'<(\w+)(\s+[^>]*?)?>', r'<\1>', desc)
            except:
                clean_desc = ''

            try:
                features = browser.find_element(by=By.CSS_SELECTOR, value='dl.ps-AttributeList').get_attribute('innerHTML')
                clean_features = re.sub(r'<(\w+)(\s+[^>]*?)?>', r'<\1>', features)
            except:
                clean_features = ''

            try:
                compatibility = browser.find_element(by=By.CSS_SELECTOR, value='.ps-VehicleCompatabilityTable').get_attribute('innerHTML').strip()
                clean_com = re.sub(r'<(\w+)(\s+[^>]*?)?>', r'<\1>', compatibility)
            except:
                clean_com = ''

            prices = browser.find_element(by=By.CSS_SELECTOR, value='.price .money').get_attribute('innerText').strip()

            images = browser.find_elements(by=By.CSS_SELECTOR, value='.product-gallery__thumbnail-list a')
            image_list = []
            for i in images:
                image = i.get_attribute('href')
                image_list.append(image)            
            
            writer.writerow([link, sku, sku2, title, vendor, clean_desc, clean_features, clean_com, prices, image_list])
            print([link, sku, sku2, title, vendor, clean_desc, clean_features, clean_com, prices, image_list])
        except Exception as e:
            print(f"{link}: {str(e).splitlines()[0]}")

['https://www.arnoldclarkautoparts.com/products/mintex-car-brake-pads-mdb1936?_pos=1&_sid=c04347c6a&_ss=r', 'MDB1934', 'MDB1936', 'Mintex Brake Pad Set fits -MercedesBenz MDB1936 (also fits other vehicles)', 'MINTEX', 'Mintex&nbsp;brake pads are manufactured by one of the largest braking suppliers, TMD Friction&nbsp;to meet and exceed OE specification products, and fully comply with European Regulation 90 (R90).&nbsp;&nbsp;With&nbsp;a 12,000&nbsp;mile&nbsp;/&nbsp;12&nbsp;month&nbsp;warranty,&nbsp;Mintex&nbsp;brake pads have an anti-squeal coating to minimise noise when slowing down.&nbsp;We always recommend using ceramic grease when fitting new brake pads. We supply&nbsp;Mintex&nbsp;Ceratec&nbsp;Brake ', '<div><dt>Brake System</dt><dd>Bosch</dd></div><div><dt>Height</dt><dd>59 mm</dd></div><div><dt>Thickness</dt><dd>16 mm</dd></div><div><dt>Wear Sensor Included</dt><dd>incl. wear warning contact</dd></div><div><dt>Width</dt><dd>187 mm</dd></div>', '<table><thead><tr><th>Position</th><t

In [2]:
csv_path = r"tmd-scrape.csv"
scrape_df = pd.read_csv(csv_path).fillna('')

final_data = []

for index0, row in scrape_df.iterrows():

    vendor = row['vendor']
    sku = row['sku']
    title = f'{vendor} ' + f'{sku} ' +row['title']
    weight = row['weight']
    ean = row['Barcode']

    sfeatures = row['clean_features']

    if sfeatures == '':
        features = ''
    else:
        features = f'<p>&nbsp;</p><h4>Features</h4>{sfeatures}'

    desc = row['clean_desc']
    if desc == '':
        desc = f'This is {title}'

    rrp = row['prices']

    fitment = row['clean_com']

    images = ast.literal_eval(row['image_list'])

    handle = (re.sub(r'[^a-zA-Z0-9\n\.]', '-', title).replace(".", "-").replace("---", "-").replace("--", "-")).lower()
    if handle.endswith('-'):
        handle = handle[:-1]

    def description(desc, features, sku):
        return f"""<h4><strong>Description</strong></h4>{desc}
        {features}
        <p>&nbsp;</p>
        <h4>Compatibility</h4>{fitment}<p>Feel free to contact us at info@mlperformance.co.uk should you wish to double check!</p>
        <p>&nbsp;</p>
        <h4>Compatibility Check</h4><p>To ensure the part(s) you have ordered fits your vehicle, we run a compatibility check prior to dispatch. We can do this either using your registration number (UK) or the last 7 digits of your VIN. Simply enter your car details prior to checkout.</p>
        <p>&nbsp;</p>
        <h4>Part Number</h4><p>{vendor}-{sku}</p>
        <p>&nbsp;</p>
        <h4>More Information</h4><p><strong>Manufactured by</strong></p><p>{vendor.title()}</p>"""

    for index, image in enumerate(images, start=1):
        info = {}
        
        info['Handle'] = handle
        info['Title'] = title
        info['Body (HTML)'] = description(desc, features, sku).replace("\n", "")
        info['Vendor'] = vendor
        info['Standardized Product Type'] = None
        info['Custom Product Type'] = None
        info['Tags'] = f"Uploaded by_Muazzim, Brand_{vendor}, Product Type_"
        info['Published'] = "TRUE"
        info['Option1 Name'] = 'Title'
        info['Option1 Value'] = 'Default Title'
        info['Option2 Name'] = None
        info['Option2 Value'] = None
        info['Option3 Name'] = None
        info['Option3 Value'] = None

        info['Variant SKU'] = f'{vendor}-' + str(sku)
        info['Variant Grams'] = str(weight*1000)
        info['Variant Inventory Tracker'] = "shopify"
        info['Variant Inventory Policy'] = 'continue'
        info['Variant Fulfillment Service'] = 'manual'
        info['Variant Price'] = None
        info['Variant Compare At Price'] = rrp
        info['Variant Requires Shipping'] = 'TRUE'
        info['Variant Taxable'] = 'TRUE'
        info['Variant Barcode'] = ean

        # For each image, create a new entry
        info['Image Src'] = image
        info['Image Position'] = index
        info['Image Alt Text'] = title
        info['Gift Card'] = None
        info['SEO Title'] = title
        info['SEO Description'] = 'Get ' + title + ' for your car to get your desired looks and performance from ML Performance at the lowest price with FREE UK shipping & next day delivery on in stock items. Very cheap prices & good service.'
        info['Google Shopping / Google Product Category'] = None
        info['Google Shopping / Gender'] = None
        info['Google Shopping / Age Group'] = None
        info['Google Shopping / MPN'] = sku
        info['Google Shopping / AdWords Grouping'] = None
        info['Google Shopping / AdWords Labels'] = None
        info['Google Shopping / Condition'] = 'new'
        info['Google Shopping / Custom Product'] = None
        info['Google Shopping / Custom Label 0'] = None
        info['Google Shopping / Custom Label 1'] = None
        info['Google Shopping / Custom Label 2'] = None
        info['Google Shopping / Custom Label 3'] = None
        info['Google Shopping / Custom Label 4'] = None
        info['Variant Image'] = None
        info['Variant Weight Unit'] = 'kg'
        info['Variant Tax Code'] = 8708949900
        info['Cost per item'] = ''
        info['Margins'] = None
        info['Price / International'] = None
        info['Compare At Price / International'] = None
        info['Status'] = 'active'

        final_data.append(info)

final_df = pd.DataFrame(final_data)

output_path = os.path.join("tmd-HTML.csv")
final_df.to_csv(output_path, index=False)

print('File saved and moved to desired folder')

File saved and moved to desired folder
